# Qwen3Loop-0.6B: GRPO Tool-Calling (fronteira)
RL com recompensas verificaveis sobre a politica V3 tool-SFT: selecao exata + stdout executado + abstencao + formato + anti-verbosidade, com ancora KL na V3. Sem tocar nas baterias holdout.

In [ ]:
!nvidia-smi

In [ ]:
# 2. Dependencias
!pip uninstall -y torchao
!pip install -q transformers datasets accelerate peft

In [ ]:
# 3. Localizar base V3, rollouts, script e engine
import os, sys, shutil
model_path, rollout_path, script_path, engine_src, probe_path = None, None, None, None, None
for root, dirs, files in os.walk('/kaggle/input'):
    if os.path.basename(root) == 'v3_model' and 'model.safetensors' in files:
        model_path = root
    if 'grpo_frontier_v1.jsonl' in files and rollout_path is None:
        rollout_path = os.path.join(root, 'grpo_frontier_v1.jsonl')
    if 'probe_25.jsonl' in files and probe_path is None:
        probe_path = os.path.join(root, 'probe_25.jsonl')
    if 'train_grpo_tool_v3.py' in files and script_path is None:
        script_path = os.path.join(root, 'train_grpo_tool_v3.py')
    if 'modeling_qwen3loop.py' in files and 'filtered_kwargs' in open(os.path.join(root, 'modeling_qwen3loop.py')).read() and engine_src is None:
        engine_src = root
print('model:', model_path)
print('rollouts:', rollout_path)
print('script:', script_path)
print('engine:', engine_src)
print('probe:', probe_path)
import pathlib
pp = pathlib.Path(probe_path)
print('probe existe:', pp.exists(), 'bytes:',
      pp.stat().st_size if pp.exists() else -1)
assert pp.exists() and pp.stat().st_size > 1000, 'probe ausente ou vazio!'
assert model_path and rollout_path and script_path and engine_src
os.makedirs('/kaggle/working/engine', exist_ok=True)
shutil.copytree(engine_src, '/kaggle/working/engine/qwen3loop_python',
                dirs_exist_ok=True)
shutil.copytree(engine_src, '/kaggle/working/qwen3loop_python',
                dirs_exist_ok=True)
sys.path.insert(0, '/kaggle/working')
sys.path.insert(0, '/kaggle/working/engine')
shutil.copy(os.path.join(os.path.dirname(script_path), 'rewards_tool_calling.py'),
            '/kaggle/working/rewards_tool_calling.py')
shutil.copy(script_path, '/kaggle/working/train_grpo_tool_v3.py')
import qwen3loop_python
print('engine ok:', qwen3loop_python.__file__)

In [ ]:
# 4. GRPO 150 steps (LoRA r16, G=4, KL 0.10 (v5-safe))
output_dir = '/kaggle/working/qwen3loop_tool_grpo_v3'
!cd /kaggle/working && python train_grpo_tool_v3.py \
    --model_dir {model_path} \
    --rollouts {rollout_path} \
    --output_dir {output_dir} \
    --steps 150 --beta_kl 0.10 --len_w 0.10 \
    --probe {probe_path}

In [ ]:
# 5. Compactar adaptador + probe p/ download
!cd /kaggle/working && zip -r qwen3loop_tool_grpo_v3.zip qwen3loop_tool_grpo_v3/ | tail -n 2
import os
final = '/kaggle/working/qwen3loop_tool_grpo_v3/adapter_model.safetensors'
assert os.path.exists(final) and os.path.getsize(final) > 10**6, 'GRPO NAO GEROU ADAPTADOR!'
print('adapters ok')